# Brisbane Parking Meter Data Aggregation

This notebook prepares Brisbane parking meter data for use as a Mobile Zone-level dimension table in Power BI.

The original meter-level dataset is aggregated so that each Mobile Zone appears only once, while preserving key supply, pricing, restriction, and location information.

In [1]:
import pandas as pd

FILE = "Cleaned_brisbane-parking-meters.xlsx"

meters = pd.read_excel(FILE)

print(meters.shape)
meters.head()

(944, 22)


,Meter_Number,Parking_System_Type,Street_Name,Suburb,Maximum_Stay_Hours,Restriction_Type,Tow_Away_Flag,Operational_Day,Operational_Time_Weekdays,Operational_Time_Saturday,...,Free_Parking_Minutes,Weekday_Rate,AfterHours_Weekend_Rates,Vehicle_Bays,Motorcycle_Bays,Motorcycle_Rate,Longitude,Latitude,Mobile_Zone,Maximum_Cap_Charge
0,4244,Ticketless Meter,Creek St,Brisbane City,3,No Stopping,No,Everyday,"7AM-4PM, 7PM-10PM",7AM-7PM,...,0,6.35,3.45,4,0,NaN,153.029667,-27.468184,4000103,NaN
1,4230,Ticketless Meter,Market St,Brisbane City,3,Taxi Zone,No,Everyday,7AM-10PM,7AM-7PM,...,0,6.35,3.45,5,0,NaN,153.029743,-27.469636,4000104,NaN
2,4231,Ticketless Meter,Charlotte St,Brisbane City,3,NaN,No,Everyday,7AM-10PM,7AM-7PM,...,0,6.35,3.45,6,0,NaN,153.029238,-27.469122,4000105,NaN
3,4232,Ticketless Meter,Charlotte St,Brisbane City,3,NaN,No,Everyday,7AM-10PM,7AM-7PM,...,0,6.35,3.45,9,0,NaN,153.028946,-27.469360,4000105,NaN
4,4260,App Only,Edward St,Brisbane City,3,Clearway / Loading Zone,No,Everyday,"10AM-2PM, 7PM-10PM",7AM-7PM,...,0,6.35,3.45,9,0,NaN,153.025750,-27.467070,4000107,NaN


## 1. Inspect Source Data

The source dataset is inspected to understand its columns, data types, and overall structure before aggregation.

In [2]:
print(meters.columns.tolist())
print()
print(meters.dtypes)

['Meter_Number', 'Parking_System_Type', 'Street_Name', 'Suburb', 'Maximum_Stay_Hours', 'Restriction_Type', 'Tow_Away_Flag', 'Operational_Day', 'Operational_Time_Weekdays', 'Operational_Time_Saturday', 'Operational_Time_Sunday', 'Tariff_Zone', 'Free_Parking_Minutes', 'Weekday_Rate', 'AfterHours_Weekend_Rates', 'Vehicle_Bays', 'Motorcycle_Bays', 'Motorcycle_Rate', 'Longitude', 'Latitude', 'Mobile_Zone', 'Maximum_Cap_Charge']

Meter_Number                   int64
Parking_System_Type           object
Street_Name                   object
Suburb                        object
Maximum_Stay_Hours            object
Restriction_Type              object
Tow_Away_Flag                 object
Operational_Day               object
Operational_Time_Weekdays     object
Operational_Time_Saturday     object
Operational_Time_Sunday       object
Tariff_Zone                   object
Free_Parking_Minutes           int64
Weekday_Rate                 float64
AfterHours_Weekend_Rates     float64
Vehicle_Bays     

## 2. Validate Mobile Zone Structure

The number of rows, unique Mobile Zones, missing keys, and duplicate zone records are checked to confirm that multiple meters can belong to the same Mobile Zone.

In [3]:
print("Total rows:", len(meters))
print("Unique Mobile Zones:", meters["Mobile_Zone"].nunique())
print("Missing Mobile Zones:", meters["Mobile_Zone"].isna().sum())
print(
    "Duplicate zone rows:",
    meters["Mobile_Zone"].duplicated().sum()
)

Total rows: 944
Unique Mobile Zones: 614
Missing Mobile Zones: 0
Duplicate zone rows: 330


## 3. Check Within-Zone Variation

Columns are checked for variation within each Mobile Zone to determine which fields can safely use a single representative value and which must preserve multiple values.

In [4]:
check_cols = [
    "Parking_System_Type",
    "Street_Name",
    "Suburb",
    "Maximum_Stay_Hours",
    "Restriction_Type",
    "Tow_Away_Flag",
    "Operational_Day",
    "Operational_Time_Weekdays",
    "Operational_Time_Saturday",
    "Operational_Time_Sunday",
    "Tariff_Zone",
    "Free_Parking_Minutes",
    "Weekday_Rate",
    "AfterHours_Weekend_Rates",
    "Vehicle_Bays",
    "Motorcycle_Bays",
    "Motorcycle_Rate",
    "Maximum_Cap_Charge"
]

variation_summary = (
    meters
    .groupby("Mobile_Zone")[check_cols]
    .nunique(dropna=True)
    .gt(1)
    .sum()
    .sort_values(ascending=False)
)

variation_summary

Vehicle_Bays                 207
Parking_System_Type           35
Street_Name                   34
Suburb                         2
Restriction_Type               1
Tow_Away_Flag                  1
Operational_Day                1
Operational_Time_Weekdays      1
Motorcycle_Bays                1
Weekday_Rate                   0
Motorcycle_Rate                0
AfterHours_Weekend_Rates       0
Operational_Time_Sunday        0
Free_Parking_Minutes           0
Tariff_Zone                    0
Operational_Time_Saturday      0
Maximum_Stay_Hours             0
Maximum_Cap_Charge             0
dtype: int64

## 4. Define Aggregation Rules

Different aggregation methods are applied according to the meaning of each field:

- meter counts are counted uniquely
- parking bays are summed
- latitude and longitude are averaged
- categorical values with variation are preserved as combined unique values
- fields that are consistent within each zone use the first value
- all suburb values are retained in Suburb_All
- where a Mobile Zone spans multiple suburbs, the suburb containing the greater number of vehicle bays is assigned as Primary_Suburb

In [5]:
def combine_unique(series):
    values = (
        series
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
    )

    return " | ".join(sorted(values))

### Assign Primary Suburb

A small number of Mobile Zones span more than one suburb.

To preserve a one-row-per-Mobile-Zone structure, the suburb containing the greater number of vehicle bays is assigned as Primary_Suburb. All original suburb values are retained separately in Suburb_All.

In [6]:
suburb_bays = (
    meters
    .groupby(
        ["Mobile_Zone", "Suburb"],
        as_index=False
    )
    .agg(
        Suburb_Vehicle_Bays=(
            "Vehicle_Bays",
            "sum"
        )
    )
)

multi_suburb_zones = (
    meters
    .groupby("Mobile_Zone")["Suburb"]
    .nunique()
)

multi_suburb_zones = multi_suburb_zones[
    multi_suburb_zones > 1
].index

suburb_bays[
    suburb_bays["Mobile_Zone"].isin(
        multi_suburb_zones
    )
]

,Mobile_Zone,Suburb,Suburb_Vehicle_Bays
76,4000230,Auchenflower,12
77,4000230,Milton,9
193,4000366,Bowen Hills,9
194,4000366,Fortitude Valley,11


In [7]:
primary_suburb = (
    suburb_bays
    .sort_values(
        [
            "Mobile_Zone",
            "Suburb_Vehicle_Bays",
            "Suburb"
        ],
        ascending=[True, False, True]
    )
    .drop_duplicates("Mobile_Zone")
    [
        ["Mobile_Zone", "Suburb"]
    ]
    .rename(
        columns={
            "Suburb": "Primary_Suburb"
        }
    )
)

primary_suburb[
    primary_suburb["Mobile_Zone"].isin(
        multi_suburb_zones
    )
]

,Mobile_Zone,Primary_Suburb
76,4000230,Auchenflower
194,4000366,Fortitude Valley


## 5. Aggregate to Mobile Zone Level

The meter-level dataset is aggregated so that each Mobile Zone is represented by exactly one row.

In [8]:
meter_zone = (
    meters
    .groupby("Mobile_Zone", as_index=False)
    .agg(
        Meter_Count=("Meter_Number", "nunique"),

        Parking_System_Type=(
            "Parking_System_Type",
            combine_unique
        ),

        Street_Name=(
            "Street_Name",
            combine_unique
        ),

       Suburb_All=(
            "Suburb",
            combine_unique
        ),

        Maximum_Stay_Hours=(
            "Maximum_Stay_Hours",
            "first"
        ),

        Restriction_Type=(
            "Restriction_Type",
            combine_unique
        ),

        Tow_Away_Flag=(
            "Tow_Away_Flag",
            combine_unique
        ),

        Operational_Day=(
            "Operational_Day",
            combine_unique
        ),

        Operational_Time_Weekdays=(
            "Operational_Time_Weekdays",
            combine_unique
        ),

        Operational_Time_Saturday=(
            "Operational_Time_Saturday",
            "first"
        ),

        Operational_Time_Sunday=(
            "Operational_Time_Sunday",
            "first"
        ),

        Tariff_Zone=(
            "Tariff_Zone",
            "first"
        ),

        Free_Parking_Minutes=(
            "Free_Parking_Minutes",
            "first"
        ),

        Weekday_Rate=(
            "Weekday_Rate",
            "first"
        ),

        AfterHours_Weekend_Rates=(
            "AfterHours_Weekend_Rates",
            "first"
        ),

        Total_Vehicle_Bays=(
            "Vehicle_Bays",
            "sum"
        ),

        Total_Motorcycle_Bays=(
            "Motorcycle_Bays",
            "sum"
        ),

        Motorcycle_Rate=(
            "Motorcycle_Rate",
            "first"
        ),

        Longitude=(
            "Longitude",
            "mean"
        ),

        Latitude=(
            "Latitude",
            "mean"
        ),

        Maximum_Cap_Charge=(
            "Maximum_Cap_Charge",
            "first"
        )
    )
)

In [9]:
meter_zone = meter_zone.merge(
    primary_suburb,
    on="Mobile_Zone",
    how="left",
    validate="one_to_one"
)

In [10]:
print(meter_zone.shape)
meter_zone.head()

(614, 23)


,Mobile_Zone,Meter_Count,Parking_System_Type,Street_Name,Suburb_All,Maximum_Stay_Hours,Restriction_Type,Tow_Away_Flag,Operational_Day,Operational_Time_Weekdays,...,Free_Parking_Minutes,Weekday_Rate,AfterHours_Weekend_Rates,Total_Vehicle_Bays,Total_Motorcycle_Bays,Motorcycle_Rate,Longitude,Latitude,Maximum_Cap_Charge,Primary_Suburb
0,4000103,1,Ticketless Meter,Creek St,Brisbane City,3,No Stopping,No,Everyday,"7AM-4PM, 7PM-10PM",...,0,6.35,3.45,4,0,NaN,153.029667,-27.468184,None,Brisbane City
1,4000104,1,Ticketless Meter,Market St,Brisbane City,3,Taxi Zone,No,Everyday,7AM-10PM,...,0,6.35,3.45,5,0,NaN,153.029743,-27.469636,None,Brisbane City
2,4000105,2,Ticketless Meter,Charlotte St,Brisbane City,3,,No,Everyday,7AM-10PM,...,0,6.35,3.45,15,0,NaN,153.029092,-27.469241,None,Brisbane City
3,4000107,1,App Only,Edward St,Brisbane City,3,Clearway / Loading Zone,No,Everyday,"10AM-2PM, 7PM-10PM",...,0,6.35,3.45,9,0,NaN,153.025750,-27.467070,None,Brisbane City
4,4000108,2,Ticketless Meter,Margaret St,Brisbane City,3,No Stopping,No,Everyday,"9AM-3PM, 7PM-10PM",...,0,6.35,3.45,16,0,NaN,153.027179,-27.473581,None,Brisbane City


## 6. Validate Aggregated Results

The aggregated table is validated to confirm that:

- each Mobile Zone appears only once
- the total number of meters is preserved
- total vehicle bays are preserved
- total motorcycle bays are preserved
- every Mobile Zone has a Primary_Suburb

In [11]:
print(
    "Duplicate Mobile Zones:",
    meter_zone["Mobile_Zone"].duplicated().sum()
)

print(
    "Meters before:",
    meters["Meter_Number"].nunique()
)

print(
    "Meters after:",
    meter_zone["Meter_Count"].sum()
)

print(
    "Vehicle bays before:",
    meters["Vehicle_Bays"].sum()
)

print(
    "Vehicle bays after:",
    meter_zone["Total_Vehicle_Bays"].sum()
)

print(
    "Motorcycle bays before:",
    meters["Motorcycle_Bays"].sum()
)

print(
    "Motorcycle bays after:",
    meter_zone["Total_Motorcycle_Bays"].sum()
)
print(
    "Missing Primary Suburbs:",
    meter_zone["Primary_Suburb"].isna().sum()
)

Duplicate Mobile Zones: 0
Meters before: 944
Meters after: 944
Vehicle bays before: 9136
Vehicle bays after: 9136
Motorcycle bays before: 256
Motorcycle bays after: 256
Missing Primary Suburbs: 0


In [12]:
meter_zone[
    meter_zone["Mobile_Zone"].isin(
        multi_suburb_zones
    )
][
    [
        "Mobile_Zone",
        "Primary_Suburb",
        "Suburb_All",
        "Total_Vehicle_Bays"
    ]
]

,Mobile_Zone,Primary_Suburb,Suburb_All,Total_Vehicle_Bays
76,4000230,Auchenflower,Auchenflower | Milton,21
192,4000366,Fortitude Valley,Bowen Hills | Fortitude Valley,20


## 7. Prepare Relationship Key for Power BI

The Mobile Zone column name is standardized so that the meter dimension can be related directly to the occupancy table in Power BI.

In [13]:
meter_zone = meter_zone.rename(
    columns={"Mobile_Zone": "MOBILE_ZONE"}
)

print(
    "Unique zones:",
    meter_zone["MOBILE_ZONE"].nunique()
)

print(
    "Duplicate zones:",
    meter_zone["MOBILE_ZONE"].duplicated().sum()
)

Unique zones: 614
Duplicate zones: 0


In [14]:
print("Final shape:", meter_zone.shape)
print()
print(meter_zone.columns.tolist())

meter_zone.head(10)

Final shape: (614, 23)

['MOBILE_ZONE', 'Meter_Count', 'Parking_System_Type', 'Street_Name', 'Suburb_All', 'Maximum_Stay_Hours', 'Restriction_Type', 'Tow_Away_Flag', 'Operational_Day', 'Operational_Time_Weekdays', 'Operational_Time_Saturday', 'Operational_Time_Sunday', 'Tariff_Zone', 'Free_Parking_Minutes', 'Weekday_Rate', 'AfterHours_Weekend_Rates', 'Total_Vehicle_Bays', 'Total_Motorcycle_Bays', 'Motorcycle_Rate', 'Longitude', 'Latitude', 'Maximum_Cap_Charge', 'Primary_Suburb']


,MOBILE_ZONE,Meter_Count,Parking_System_Type,Street_Name,Suburb_All,Maximum_Stay_Hours,Restriction_Type,Tow_Away_Flag,Operational_Day,Operational_Time_Weekdays,...,Free_Parking_Minutes,Weekday_Rate,AfterHours_Weekend_Rates,Total_Vehicle_Bays,Total_Motorcycle_Bays,Motorcycle_Rate,Longitude,Latitude,Maximum_Cap_Charge,Primary_Suburb
0,4000103,1,Ticketless Meter,Creek St,Brisbane City,3,No Stopping,No,Everyday,"7AM-4PM, 7PM-10PM",...,0,6.35,3.45,4,0,NaN,153.029667,-27.468184,None,Brisbane City
1,4000104,1,Ticketless Meter,Market St,Brisbane City,3,Taxi Zone,No,Everyday,7AM-10PM,...,0,6.35,3.45,5,0,NaN,153.029743,-27.469636,None,Brisbane City
2,4000105,2,Ticketless Meter,Charlotte St,Brisbane City,3,,No,Everyday,7AM-10PM,...,0,6.35,3.45,15,0,NaN,153.029092,-27.469241,None,Brisbane City
3,4000107,1,App Only,Edward St,Brisbane City,3,Clearway / Loading Zone,No,Everyday,"10AM-2PM, 7PM-10PM",...,0,6.35,3.45,9,0,NaN,153.025750,-27.467070,None,Brisbane City
4,4000108,2,Ticketless Meter,Margaret St,Brisbane City,3,No Stopping,No,Everyday,"9AM-3PM, 7PM-10PM",...,0,6.35,3.45,16,0,NaN,153.027179,-27.473581,None,Brisbane City
5,4000110,1,Ticketless Meter,Charlotte St,Brisbane City,12,Motorcycle Only,No,Weekdays,7AM-7PM,...,0,2.20,NaN,0,38,2.2,153.026400,-27.471355,11.5,Brisbane City
6,4000111,2,Hybrid | Ticketless Meter,Charlotte St,Brisbane City,3,,No,Everyday,7AM-10PM,...,0,6.35,3.45,20,0,NaN,153.026499,-27.471279,None,Brisbane City
7,4000112,2,App Only | Ticketless Meter,George St,Brisbane City,3,Clearway,Yes,Everyday,"9AM-4PM, 7PM-10PM",...,0,6.35,3.45,11,0,NaN,153.025777,-27.473136,None,Brisbane City
8,4000113,1,Ticketless Meter,Mary St,Brisbane City,3,,No,Everyday,7AM-10PM,...,0,6.35,3.45,5,0,NaN,153.026606,-27.472513,None,Brisbane City
9,4000114,2,Ticketless Meter,Mary St,Brisbane City,3,,No,Everyday,7AM-10PM,...,0,6.35,3.45,10,0,NaN,153.028384,-27.471218,None,Brisbane City


## 8. Export Mobile Zone Dimension Table

The validated Mobile Zone-level table is exported as a CSV file for use as a dimension table in Power BI.

In [15]:
meter_zone.to_csv(
    "parking_meter_by_mobile_zone.csv",
    index=False
)

print("File exported successfully.")

File exported successfully.
